# Gradio web interface for GitHub Issues

This lab starts from the solution of the *Command-line interface for GitHub Issues* lab: `ghcli list` and `ghcli create` list and create a repository's issues, and are tested. We are going to add a web interface to them, with Gradio.

The code is the `gradio-en` branch of the [shuuchuu/ghcli](https://github.com/shuuchuu/ghcli/tree/gradio-en) repository, which the notebook clones. `src/ghcli/gui.py` is the interface to complete: `build_app` builds an app with two tabs, empty for now, and `_format_issue` writes an issue in Markdown. A file is written with a cell that starts with `%%writefile <file>`, which replaces it with the rest of its content, or in the file browser, on the left (a double click opens it in an editor). The solutions are also on the repository's [`gradio-solution-en` branch](https://github.com/shuuchuu/ghcli/tree/gradio-solution-en).

## Setting up the working environment

Run the following cell: it clones the repository into the `ghcli/` folder, installs its dependencies, and makes its package importable.

In [ ]:
import pathlib
import sys

root_dir = pathlib.Path.cwd()
if root_dir.name != "ghcli":
  root_dir /= "ghcli"
if not root_dir.exists():
  !git clone -b gradio-en https://github.com/shuuchuu/ghcli.git {root_dir}
%cd {root_dir}
!uv pip install gradio niquests
if str(root_dir / "src") not in sys.path:
  sys.path.insert(0, str(root_dir / "src"))

`ghcli` needs a GitHub token to create an issue: if you don't have one, create a test repository and a token as [the command-line interface's instructions](https://github.com/shuuchuu/ghcli/tree/main-en#a-test-repository-and-a-github-token) explain. Write your GitHub user name in the following cell, between the quotes, then run it: it asks for the token without showing it, and puts it in the `GHCLI_TOKEN` environment variable, where `ghcli` reads it. Never write a token in a cell: it would stay on the screen and in every copy of the notebook.

Without a GitHub account, leave `github_user` empty: a public repository's issues are listed without a token (the API then accepts 60 requests per hour and per address, which a room behind one address shares), but no issue can be created.

In [ ]:
import os
from getpass import getpass

github_user = ""
if github_user:
  os.environ["GHCLI_TOKEN"] = getpass("GitHub token: ")

The following cell starts the interface: Colab shows it below the cell, and gives a public link (`*.gradio.live`) that opens it in another tab. Run it again after each change to `gui.py`: it reads the file again, and first closes the interfaces started before it (`gr.close_all()`): open the new link, the old one no longer answers.

In [ ]:
import importlib

import gradio as gr

import ghcli.gui

# Reads gui.py again, after a change.
importlib.reload(ghcli.gui)
# Closes the interfaces of the previous launches.
gr.close_all()
ghcli.gui.build_app().launch()

## Listing a repository's issues

In `build_app`'s *List issues* tab, add two text boxes, the repository's owner and its name, a *List the issues* button and a Markdown area where the issues show. The button calls a `_list_issues(owner: str, repo: str) -> str` function, to write, which returns the repository's issues in Markdown, separated by a horizontal rule (`---`), `No issue.` when it has none, or an error message when the API returns one.

**Useful documentation**

- https://www.gradio.app/guides/blocks-and-event-listeners
- https://www.gradio.app/docs/gradio/textbox
- https://www.gradio.app/docs/gradio/markdown

In [ ]:
# Your code here

### Additional hints

Inside a `with gr.Tab(...)` block, each component created is added to the tab, and `with gr.Row():` lays out several side by side. `button.click(function, inputs=[...], outputs=...)` calls the function with the input components' values, and shows what it returns in the output one. `list_issues` raises a `niquests.HTTPError` for an unknown repository. On `octocat` / `Hello-World`, a public repository the API reads without a token, the area shows a list of linked titles, separated by rules; on an unknown repository, `Could not list the issues: 404 Client Error: Not Found…`.

```python
def _list_issues(owner: str, repo: str) -> str:
  try:
    issues = list_issues(owner, repo)
  except HTTPError as error:
    return ...
  return "\n\n---\n\n".join(...) or "No issue."


# In build_app:
with gr.Tab("List issues"):
  with gr.Row():
    owner = gr.Textbox(label="Owner", placeholder="octocat")
    ...
  button = gr.Button("List the issues")
  issues = gr.Markdown()
  button.click(_list_issues, inputs=[...], outputs=...)
```

### Solution

In [ ]:
%%writefile src/ghcli/gui.py
"""Web interface to GitHub issues, with Gradio."""

import gradio as gr
from niquests import HTTPError

from .api import list_issues
from .model import Issue


def _format_issue(issue: Issue) -> str:
    """Show an issue in Markdown: its title, linked to its page, then its body.

    Args:
        issue: The issue to show.

    Returns:
        Its Markdown.
    """
    return f"### [{issue.title}]({issue.url})\n\n{issue.body}"


def _list_issues(owner: str, repo: str) -> str:
    try:
        issues = list_issues(owner, repo)
    except HTTPError as error:
        return f"Could not list the issues: {error}"
    return "\n\n---\n\n".join(map(_format_issue, issues)) or "No issue."


def build_app() -> gr.Blocks:
    """Build the interface: a tab lists a repository's issues, another creates one.

    Returns:
        The app, which `launch()` serves.
    """
    with gr.Blocks(title="ghcli") as app:
        gr.Markdown("# ghcli\n\nList and create the issues of a GitHub repository.")
        with gr.Tab("List issues"):
            with gr.Row():
                owner = gr.Textbox(label="Owner", placeholder="octocat")
                repo = gr.Textbox(label="Repository", placeholder="Hello-World")
            button = gr.Button("List the issues")
            issues = gr.Markdown()
            # The function's arguments are the inputs' values, its return value
            # the output's.
            button.click(_list_issues, inputs=[owner, repo], outputs=issues)
        with gr.Tab("Create an issue"):
            # Your code here
            ...
    return app


def create_gui() -> None:
    """Serve the interface, at http://127.0.0.1:7860 by default."""
    build_app().launch()

In [ ]:
import importlib

import gradio as gr

import ghcli.gui

# Reads gui.py again, after a change.
importlib.reload(ghcli.gui)
# Closes the interfaces of the previous launches.
gr.close_all()
ghcli.gui.build_app().launch()

## Creating an issue

In the *Create an issue* tab, add the repository's owner, its name, the issue's title and its body (a multi-line text box), a *Create the issue* button and a Markdown area that shows the created issue, or an error message. The button calls a `_create_issue` function, to write. Without a token (`GHCLI_TOKEN` empty), it returns a message that says so, without calling `create_issue`, which would then stop the whole program: that is what a command line should do, not an interface. Create an issue in your test repository, then list its issues in the other tab.

In [ ]:
# Your code here

### Additional hints

`gr.Textbox(label="Body", lines=6)` is a six-line text box. `os.environ.get("GHCLI_TOKEN")` is `None` or `""` without a token (`import os` at the top of `gui.py`). The two tabs can reuse the same variable names: each `click` gets the components that exist when it's called. With a token, the created issue shows, and the other tab lists it; without one, `_create_issue`'s message.

```python
def _create_issue(owner: str, repo: str, title: str, body: str) -> str:
  if not os.environ.get("GHCLI_TOKEN"):
    return ...
  try:
    issue = create_issue(owner, repo, title, body)
  except HTTPError as error:
    return ...
  return _format_issue(issue)
```

### Solution

In [ ]:
%%writefile src/ghcli/gui.py
"""Web interface to GitHub issues, with Gradio."""

import os

import gradio as gr
from niquests import HTTPError

from .api import create_issue, list_issues
from .model import Issue


def _format_issue(issue: Issue) -> str:
    """Show an issue in Markdown: its title, linked to its page, then its body.

    Args:
        issue: The issue to show.

    Returns:
        Its Markdown.
    """
    return f"### [{issue.title}]({issue.url})\n\n{issue.body}"


def _list_issues(owner: str, repo: str) -> str:
    try:
        issues = list_issues(owner, repo)
    except HTTPError as error:
        return f"Could not list the issues: {error}"
    return "\n\n---\n\n".join(map(_format_issue, issues)) or "No issue."


def _create_issue(owner: str, repo: str, title: str, body: str) -> str:
    if not os.environ.get("GHCLI_TOKEN"):
        return (
            "Could not create the issue without a GitHub token: set the "
            "GHCLI_TOKEN environment variable."
        )
    try:
        issue = create_issue(owner, repo, title, body)
    except HTTPError as error:
        return f"Could not create the issue: {error}"
    return _format_issue(issue)


def build_app() -> gr.Blocks:
    """Build the interface: a tab lists a repository's issues, another creates one.

    Returns:
        The app, which `launch()` serves.
    """
    with gr.Blocks(title="ghcli") as app:
        gr.Markdown("# ghcli\n\nList and create the issues of a GitHub repository.")
        with gr.Tab("List issues"):
            with gr.Row():
                owner = gr.Textbox(label="Owner", placeholder="octocat")
                repo = gr.Textbox(label="Repository", placeholder="Hello-World")
            button = gr.Button("List the issues")
            issues = gr.Markdown()
            # The function's arguments are the inputs' values, its return value
            # the output's.
            button.click(_list_issues, inputs=[owner, repo], outputs=issues)
        with gr.Tab("Create an issue"):
            with gr.Row():
                owner = gr.Textbox(label="Owner")
                repo = gr.Textbox(label="Repository")
            title = gr.Textbox(label="Title")
            body = gr.Textbox(label="Body", lines=6)
            button = gr.Button("Create the issue")
            issue = gr.Markdown()
            button.click(
                _create_issue, inputs=[owner, repo, title, body], outputs=issue
            )
    return app


def create_gui() -> None:
    """Serve the interface, at http://127.0.0.1:7860 by default."""
    build_app().launch()

In [ ]:
import importlib

import gradio as gr

import ghcli.gui

# Reads gui.py again, after a change.
importlib.reload(ghcli.gui)
# Closes the interfaces of the previous launches.
gr.close_all()
ghcli.gui.build_app().launch()